# 🧠 CerebroSpheroid V5: Cloud TPU JAX High-Throughput Engine
### Progressive Cortical Expansion (100M ➔ 166.3M Compact) & 200B Token Campaign • Google Colab TPU v5e-1 / v6e

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_TPU_JAX.ipynb)

### **High-Throughput JAX Pre-Training & Reasoning Architecture:**
1. **100M Foundation Model Resumption (Step 920,000 Base)**:
   - Resumes seamlessly from the Step 920k foundation model (`v5_small_latest_step920k.pt`) preserving all 15B tokens of foundational linguistic manifolds.
   - Automatically adapts weights into `v5_compact_32shells` (166.3M params, 32 concentric shells, 96 shortcut tracts) with Net2Net bit-for-bit zero-regression mathematical invariance ($\Delta \text{logits} = 0.000000\text{e}+00$).
2. **Progressive Encephalization Warmup ($\alpha: 0.0 \to 1.0$)**:
   - Motor readout transitions smoothly from Core Shell 15 to Outer Mantle Shell 31 over 1,000 steps (`--encephalization_warmup_steps 1000`), preventing early loss spikes.
3. **Cyclical Core Freezing & Consolidation Annealing**:
   - Core weights in Shells 0–15 remain strictly frozen for 900 steps while the outer mantle learns, then unfreeze for 100 steps of gentle annealing ($LR_{\text{core}} = 0.1 \times LR_{\text{mantle}}$) every 1,000 steps (`--freeze_core`).
4. **High-Density Teacher Digested Streaming**:
   - Streams pre-computed dynamic next-token information density weights directly from `AjitKumar-Bio/cosmopedia-v2-digested` (or 4-phase developmental curriculum via `--dataset_source autonomous`).
5. **Automated Held-Out Loss Convergence Stopping ($\le 1.60$)**:
   - Evaluates held-out validation loss across all 8 canonical domains every 100 steps.
   - Requires **3 consecutive evaluations** $\le 1.60$ to guarantee true convergence before saving `best.pt` and cleanly halting.
6. **Scale & Throughput on TPU v5e-1 / v6e**:
   - Global effective batch size of **256 sequences** (131,072 tokens per optimizer step).
   - Sustained execution rate: **~200,000–350,000 tok/s** via systolic MXU units and XLA JIT compilation.
7. **Real-time Remote Tracking**:
   - Live token ingestion, loss curves, and checkpoint sync tracked continuously on `bioflo.in`.

---
### ⚡ Colab Runtime Settings:
- **Runtime** -> **Change runtime type** -> **TPU v5e** (TPU v5e-1)


## Step 1: Verify Google Colab Cloud TPU v5e-1 & JAX Environment


In [ ]:
import jax
import jax.numpy as jnp

print(f"JAX Version      : {jax.__version__}")
print(f"JAX Default Dev  : {jax.default_backend().upper()}")
devices = jax.devices()
print(f"Detected Devices : {devices}")
dev_kind = devices[0].device_kind if devices else "None"
print(f"Device type      : {dev_kind}")

is_tpu = any("TPU" in d.device_kind.upper() for d in devices)
if is_tpu:
    print(f"\n✅ Successfully connected to Cloud TPU! ({dev_kind}, {len(devices)} core(s) active)")
    if "v6" in dev_kind.lower() or "trillium" in dev_kind.lower():
        print("⚡ TPU v6e (Trillium 32GB HBM) active: ~200,000–350,000 tok/s native BF16 MXU acceleration enabled!")
    elif "v5" in dev_kind.lower() or "lite" in dev_kind.lower():
        print("⚡ TPU v5e-1 active: ~50,000–120,000 tok/s native BF16 MXU acceleration enabled.")
    else:
        print(f"⚡ TPU active ({dev_kind}).")
else:
    print("\n⚠️ Running on CPU/GPU. To enable TPU v5e-1, go to Runtime -> Change runtime type -> TPU v5e.")


## Step 2: Initialize Local Scratch Checkpoint Storage & Bioflo.in Sync
Checkpoints are saved locally on the Colab instance and mirrored to `bioflo.in` in the background (no Google Drive mount needed).


In [ ]:
import os, urllib.request, json

CKPT_DIR = "/content/checkpoints" if os.path.exists("/content") else os.path.abspath("checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"📁 Local scratch checkpoint directory: {CKPT_DIR}")

# Select target preset: 'v5_compact_32shells' (166.3M expanded), 'v5_small' (100M base), or 'v5_base' (500M)
MODEL_PRESET = "v5_compact_32shells"
norm_model = "v5_compact" if "compact" in MODEL_PRESET else ("v5_base" if "base" in MODEL_PRESET else "v5_small")

# Optional: Auto-detect Google Drive if mounted for persistent mirror
if os.path.exists("/content/drive/MyDrive"):
    DRIVE_DIR = "/content/drive/MyDrive/cerebro_checkpoints"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"💾 Google Drive mirror active: {DRIVE_DIR}")

# Verify Bioflo.in synchronization endpoint status for MODEL_PRESET
try:
    req = urllib.request.Request(
        f"https://bioflo.in/api/cerebro/sync.php?action=status&model={norm_model}",
        headers={"User-Agent": "CerebroColabTPU/5.0"}
    )
    with urllib.request.urlopen(req, timeout=5) as resp:
        st = json.loads(resp.read().decode("utf-8"))
        print(f"✓ bioflo.in connection OK: model='{norm_model}' | active={st.get('active')} | latest_step={st.get('latest_step', 'None')}")
except Exception as e:
    print(f"ℹ️ bioflo.in sync notice: {e}")


## Step 3: Install Dependencies, Authenticate Hugging Face & Synchronize Repository
1. **Installs dependencies**: `transformers`, `datasets`, `sentencepiece`, `optax`, `torch`, `pyarrow`, `huggingface_hub`.
2. **Authenticates Hugging Face**: Eliminates HTTP 429 rate limiting on shared Colab cloud IPs using your auth token, enabling clean, resilient online streaming from the full Hugging Face corpora.
3. **Synchronizes repository**: Pulls latest `cerebro_spheroid_v5` codebase.

In [ ]:
# ==============================================================================
# 📦 Step 3: Dependencies, Hugging Face Hub Auth & Repository Sync
# ==============================================================================
!pip install -q transformers datasets sentencepiece optax torch pyarrow huggingface_hub

import os, sys, subprocess

# 1. 🔑 Authenticate Hugging Face Hub (Eliminates 429 Too Many Requests on shared Colab IPs)
try:
    from google.colab import userdata
    hf_tok = userdata.get('HF_TOKEN')
except Exception:
    hf_tok = None
if not hf_tok:
    hf_tok = os.environ.get('HF_TOKEN', '')

os.environ['HF_TOKEN'] = hf_tok
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_tok
user_hf = '/tmp/hf_cache'
os.makedirs(user_hf, exist_ok=True)
os.environ['HF_HOME'] = user_hf
os.environ['HF_HUB_CACHE'] = os.path.join(user_hf, 'hub')

try:
    from huggingface_hub import login
    login(token=hf_tok, add_to_git_credential=False)
    print('🔑 [Hugging Face] Authenticated successfully as AjitKumar-Bio!')
except Exception as e:
    print(f'⚠️ Hugging Face auth notice: {e}')

# 2. 🔄 Synchronize CerebroSpheroid V5 Repository
REPO_DIR = '/content/cerebro_v5'
if not os.path.exists(REPO_DIR):
    print('📥 Cloning cerebro_spheroid_v5 into /content/cerebro_v5...')
    subprocess.run(['git', 'clone', 'https://github.com/AjitKumar-git/cerebro_spheroid_v5.git', REPO_DIR], check=True)
else:
    print('🔄 Pulling latest updates from origin/main...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)

%cd /content/cerebro_v5
rev = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
print(f'✅ Repository synchronized at commit [{rev}]!')


## Step 4: Launch Progressive Cortical Expansion on Cloud TPU (100M ➔ 166.3M Compact)
- **Target Model**: `v5_compact_32shells` (166.3M Parameters, 32 concentric shells, $d=640$, $r=256$, 96 tracts)
- **Foundation Checkpoint**: Resumes from `v5_small_latest_step920k.pt` (Step 920,000 foundation base)
- **Net2Net Parity**: Zero-regression step 0 equivalence ($\Delta \text{logits} = 0.000000\text{e}+00$)
- **Encephalization Warmup**: 1,000 steps progressive readout transfer from Shell 15 to Shell 31
- **Cyclical Consolidation**: 900 steps core frozen / 100 steps anneal unfreeze (`--freeze_core`)
 (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)- (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)  (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)* (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)* (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)D (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)a (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)t (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)a (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)  (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)S (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)o (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)u (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)r (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)c (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)* (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)* (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code): (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)  (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)H (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)g (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)h (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)- (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)d (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)n (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)s (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)t (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)y (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)  (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)t (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)a (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)c (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)h (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)r (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)  (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)d (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)g (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)s (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)t (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)d (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)  (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)s (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)t (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)r (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)a (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)m (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)n (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)g (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)  (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)( (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)` (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)A (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)j (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)t (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)K (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)u (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)m (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)a (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)r (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)- (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)B (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)o (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)/ (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)c (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)o (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)s (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)m (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)o (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)p (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)d (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)a (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)- (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)v (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)2 (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)- (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)d (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)i (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)g (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)s (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)t (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)e (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)d (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)` (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)) (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)
 (FineWeb-Edu $\ge 3.0$, Cosmopedia v2, FineMath, Code)- **Effective Batch Size**: **256 sequences** (131,072 tokens/update) for low gradient variance
- **Stopping Rule**: Automatically halts when composite held-out validation loss across all 8 canonical domains consistently achieves $\le 1.60$ (over 3 consecutive evaluations) or hits token budget


In [ ]:
import os, sys, subprocess

# ===========================================================================
# 🧠 CerebroSpheroid V5: Progressive Cortical Expansion TPU Training Engine
# ===========================================================================
# Model Architecture : v5_compact_32shells (166.3M params, 32 shells, 96 shortcut tracts)
# Base Foundation    : v5_small_latest_step920k.pt (100M base, preserving 100% of priors)
# Mathematical Init  : Net2Net zero-regression warm-start (delta logits = 0.00e+00)
# Encephalization    : Smooth motor readout ramp (alpha: 0.0 -> 1.0) over 1,000 steps
# Consolidation      : 900 steps core frozen / 100 steps anneal every 1,000 steps
# Data Streaming     : Online streaming from FineWeb-Edu >= 3.0 & foundational curricula ('autonomous')
# Hardware Platform  : Google Colab TPU v5e-1 / v6e (Trillium) Native BF16 MXU Systolic Units
# ===========================================================================

MODEL_PRESET               = 'v5_compact_32shells'  # 166.3M parameters, 32 shells
INIT_FROM_CHECKPOINT       = None                   # Set to None: clean scratch start (zero foundation copying)
FROM_SCRATCH               = True                   # Initialize clean starting model from scratch at Step 0
RESET_STEP_COUNTER         = False                  # Preserves step counter on resumption
FREEZE_CORE                = True                   # 900-step core freeze / 100-step anneal
FREEZE_UNTIL_STEP          = 50000                  # Core frozen for first 50,000 steps, permanently unfrozen thereafter
CYCLE_LENGTH               = 1000
ANNEAL_STEPS               = 100
ENCEPHALIZATION_WARMUP     = 1000                   # 1,000 steps smooth transition
DATASET_SOURCE             = 'autonomous'                           # 4-phase developmental curriculum (FineWeb-Edu, Cosmo-v2, Math, Code)
MICRO_BATCH_SIZE           = 8                      # Micro-batch per MXU core
MAX_MICRO_BATCH_SIZE       = 16
EFFECTIVE_BATCH_SIZE       = 256                    # 256 sequences (131,072 tokens/step)
PROMPT_LEN                 = 256
SEQ_LEN                    = 512
TOTAL_STEPS                = 50000
TARGET_TOKENS              = 200_000_000_000
TARGET_EVAL_LOSS           = 1.60
CONSISTENT_EVAL_THRESHOLD  = 3
LR                         = 0.00020
MIN_LR                     = 2.5e-5
AUTO_TUNE                  = False
FORCE_TUNE                 = False
TUNE_OBJECTIVE             = 'tokens_per_sec'
LR_SCHEDULE                = 'cosine'
ELASTIC_WINDOW             = True
LOSS_WINDOW_GOVERNOR       = True
LOSS_CEILING               = 8.5
LOSS_FLOOR                 = 0.4
DELIBERATION_MODE          = 'teacher_forcing'
MIN_SAMPLING_P             = 1.00
SCHEDULE_STEPS             = 0
PITFALL_PROB               = 0.15
RESUME_TARGET              = 'latest'

# 1. Ensure repository is up to date with latest vectorized JAX code
REPO_DIR = '/content/cerebro_v5'
if not os.path.exists(REPO_DIR):
    print('📥 Cloning cerebro_spheroid_v5 into /content/cerebro_v5...')
    subprocess.run(['git', 'clone', 'https://github.com/AjitKumar-git/cerebro_spheroid_v5.git', REPO_DIR], check=True)
else:
    print('🔄 Pulling latest updates from origin/main...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)

rev = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
print(f'✅ Repository synchronized at commit [{rev}]!')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# 2. Invalidate import caches so newly pulled code takes immediate effect
import importlib
for mod in list(sys.modules.keys()):
    if any(mod.startswith(pkg) or mod == pkg for pkg in ('src', 'training', 'scripts')):
        del sys.modules[mod]
importlib.invalidate_caches()

# 3. Verify Offline Bundled 32,000 LLaMA Tokenizer
tok_dir = os.path.join(REPO_DIR, 'data', 'tokenizer')
if os.path.exists(os.path.join(tok_dir, 'tokenizer.json')):
    print(f'🔤 Offline LLaMA-32k tokenizer verified at {tok_dir} (strictly 32k vocab, zero GPT-2)!')
else:
    print('⚠️ Warning: data/tokenizer not found; git pull may be incomplete.')

# 4. Ensure HF_HOME cache is writable
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = ''
if 'HF_HOME' not in os.environ:
    os.environ['HF_HOME'] = '/tmp/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

from training.train_v5_tpu_jax import run_training

# 🚀 Launch Progressive Cortical Expansion TPU JAX Engine:
# - Net2Net zero-regression warm-start seeding 166.3M params from 100M base
# - Encephalization warmup smoothly shifting control to outer mantle
# - Cyclical core freeze & consolidation annealing (900/100)
# - Direct high-density streaming from Cosmopedia v2 Digested
run_training(
    preset=MODEL_PRESET,
    batch_size=MICRO_BATCH_SIZE,
    max_batch_size=MAX_MICRO_BATCH_SIZE,
    auto_tune=AUTO_TUNE,
    dataset_source=DATASET_SOURCE,
    lr_schedule=LR_SCHEDULE,
    optimizer='spheroid',
    force_tune=FORCE_TUNE,
    tune_objective=TUNE_OBJECTIVE,
    deliberation_mode=DELIBERATION_MODE,
    min_sampling_p=MIN_SAMPLING_P,
    schedule_steps=SCHEDULE_STEPS,
    pitfall_prob=PITFALL_PROB,
    from_scratch=FROM_SCRATCH,
    resume_target=RESUME_TARGET,
    loss_window_governor=LOSS_WINDOW_GOVERNOR,
    loss_ceiling=LOSS_CEILING,
    loss_floor=LOSS_FLOOR,
    dtype='auto',
    total_steps=TOTAL_STEPS,
    prompt_len=PROMPT_LEN,
    seq_len=SEQ_LEN,
    total_window=PROMPT_LEN + SEQ_LEN,
    elastic_window=ELASTIC_WINDOW,
    lr=LR,
    min_lr=MIN_LR,
    resume_warmup_steps=250,
    save_interval=250,
    eval_interval=100,
    cloud_sync=True,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    target_tokens=TARGET_TOKENS,
    reset_step_counter=RESET_STEP_COUNTER,
    init_from_checkpoint=INIT_FROM_CHECKPOINT,
    target_eval_loss=TARGET_EVAL_LOSS,
    consistent_eval_threshold=CONSISTENT_EVAL_THRESHOLD,
    checkpoint_dir='/content/checkpoints',
    freeze_core=FREEZE_CORE,
    freeze_until_step=FREEZE_UNTIL_STEP,
    cycle_length=CYCLE_LENGTH,
    anneal_steps=ANNEAL_STEPS,
    encephalization_warmup_steps=ENCEPHALIZATION_WARMUP,
)


### ⚡ Alternative: Launch Phase 3 via Standalone CLI
You can also execute the Phase 3 runner as a background terminal command or single cell:


In [ ]:
# ==============================================================================
# ⚡ OPTIONAL ALTERNATIVE: Standalone CLI Launch (Commented out for 'Run All')
# ==============================================================================
# The primary training runner is executed in Step 4 (Cell 8) above.
# If you prefer running via standalone bash CLI instead of programmatic execution,
# comment out Cell 8 and uncomment the lines below:
#
# !git -C /content/cerebro_v5 fetch origin main && git -C /content/cerebro_v5 reset --hard origin/main
# !python3 training/train_v5_tpu_jax.py \
#     --preset v5_compact_32shells \
#     --from_scratch \
#     --freeze_core \
#     --freeze_until_step 50000 \
#     --cycle_length 1000 \
#     --anneal_steps 100 \
#     --encephalization_warmup_steps 1000 \
#     --dataset_source "autonomous" \
#     --effective-batch-size 256 \
#     --target-tokens 200000000000 \
#     --target-eval-loss 1.60 \
#     --consistent-eval-threshold 3 \
#     --checkpoint-dir "/content/checkpoints" \
#     --cloud-sync


## Step 5: Inspect Local Checkpoints & Remote Manifest


In [ ]:
!ls -lh /content/checkpoints
import json
from src.cloud_sync import check_remote_manifest
print("\n🌐 Remote bioflo.in manifest for v5_compact:")
print(json.dumps(check_remote_manifest(model="v5_compact"), indent=2))
print("\n🌐 Remote bioflo.in manifest for v5_small:")
print(json.dumps(check_remote_manifest(model="v5_small"), indent=2))
